In [1]:
print("hello")

hello


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma

In [3]:
loader=PyPDFLoader("atomic_summary.pdf")
pages=loader.load_and_split()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=50)
chunks=text_splitter.split_documents(pages)
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")



In [5]:
vectorstore=Chroma.from_documents(documents=chunks,embedding=embeddings)

In [6]:
retriever=vectorstore.as_retriever()

In [7]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [8]:
llm=ChatOpenAI()
template="""System: you are a question anser bot.
Be factual in your response.
Respond to the follwoing questio:{question} only from 
the below context:{context},
if you do not know the answer , just say you do not know.
"""
prompt=PromptTemplate.from_template(template)

In [9]:
chain=(
    {"context":retriever|format_docs,"question":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

In [10]:
chain.invoke("what is the secret of self control")

'The secret of self-control, as outlined in Chapter 7, is to spend less time in tempting situations and avoid temptation rather than trying to resist it. The practical way to eliminate a bad habit is to reduce exposure to the trigger that causes it.'